# Fusion des sources de données — Projet Data Warehouse Films

**Fichiers en entrée :**
- `cleaned_movies.csv` — données Kaggle nettoyées
- `api_cleaned.csv` — données enrichies via TMDB API
- `metacritic_scores.csv` — scores scrappés depuis Metacritic

**Fichier en sortie :**
- `merged_movies.csv` — source unique pour l'ETL pygramETL

In [2]:
import pandas as pd

## 1. Chargement des trois fichiers

In [3]:
# Charger le fichier principal nettoyé (source Kaggle)
kaggle_df = pd.read_csv('cleaned_movies.csv')
print('Forme kaggle_df :', kaggle_df.shape)
print(kaggle_df.head(5))

Forme kaggle_df : (3181, 30)
   tmdb_id                title       original_title      director  \
0    10764    Quantum of Solace    Quantum of Solace  Marc Forster   
1    20662           Robin Hood           Robin Hood  Ridley Scott   
2     2268   The Golden Compass   The Golden Compass   Chris Weitz   
3      558         Spider-Man 2         Spider-Man 2     Sam Raimi   
4    62211  Monsters University  Monsters University   Dan Scanlon   

  primary_genre                          all_genres original_language  \
0     Adventure  Adventure, Action, Thriller, Crime                EN   
1        Action                   Action, Adventure                EN   
2     Adventure                  Adventure, Fantasy                EN   
3        Action          Action, Adventure, Fantasy                EN   
4     Animation                   Animation, Family                EN   

  primary_language  runtime  \
0          English    106.0   
1          English    140.0   
2         Íslenska

In [4]:
# Charger les données enrichies via l'API TMDB
api_df = pd.read_csv('api_cleaned.csv')
print('Forme api_df :', api_df.shape)
print(api_df.head(5))

Forme api_df : (300, 8)
   tmdb_id                                     title  runtime language  \
0    19995                                    Avatar      162       EN   
1      285  Pirates of the Caribbean: At World's End      169       EN   
2   206647                                   Spectre      148       EN   
3    49026                     The Dark Knight Rises      165       EN   
4    49529                               John Carter      132       EN   

                    studio studio_country release_date    status  
0       Dune Entertainment             US   2009-12-16  Released  
1  Jerry Bruckheimer Films             US   2007-05-19  Released  
2      Metro-Goldwyn-Mayer             US   2015-10-26  Released  
3                  Syncopy             GB   2012-07-17  Released  
4     Walt Disney Pictures             US   2012-03-07  Released  


In [5]:
# Charger les scores scrappés depuis Metacritic
meta_df = pd.read_csv('metacritic_scores.csv')
print('Forme meta_df :', meta_df.shape)
print(meta_df.head(5))

Forme meta_df : (4, 3)
              slug  metascore  user_score
0  the-dark-knight        NaN         NaN
1        inception        NaN         NaN
2     interstellar        NaN         NaN
3           avatar        NaN         NaN


## 2. Inspection des colonnes de chaque source

In [6]:
print('Colonnes Kaggle :')
print(kaggle_df.columns.tolist())

Colonnes Kaggle :
['tmdb_id', 'title', 'original_title', 'director', 'primary_genre', 'all_genres', 'original_language', 'primary_language', 'runtime', 'top_cast', 'overview', 'studio_name', 'studio_country', 'primary_country', 'release_date', 'release_year', 'release_month', 'release_quarter', 'release_day', 'release_week', 'release_season', 'budget', 'revenue', 'profit', 'roi', 'is_profit', 'revenue_tier', 'user_score', 'vote_count', 'popularity']


In [7]:
print('Colonnes API :')
print(api_df.columns.tolist())

Colonnes API :
['tmdb_id', 'title', 'runtime', 'language', 'studio', 'studio_country', 'release_date', 'status']


In [8]:
print('Colonnes Metacritic :')
print(meta_df.columns.tolist())

Colonnes Metacritic :
['slug', 'metascore', 'user_score']


## 3. Préparation des clés de jointure

On fusionne sur `tmdb_id`. On s'assure que la colonne existe et est du même type dans chaque fichier.

In [9]:
# Vérifier que tmdb_id existe dans chaque dataframe
print('tmdb_id dans kaggle_df :', 'tmdb_id' in kaggle_df.columns)
print('tmdb_id dans api_df    :', 'tmdb_id' in api_df.columns)
print('tmdb_id dans meta_df   :', 'tmdb_id' in meta_df.columns)

tmdb_id dans kaggle_df : True
tmdb_id dans api_df    : True
tmdb_id dans meta_df   : False


In [10]:
# Forcer tmdb_id en entier dans tous les fichiers pour éviter les erreurs de jointure
kaggle_df['tmdb_id'] = pd.to_numeric(kaggle_df['tmdb_id'], errors='coerce').astype('Int64')
api_df['tmdb_id']    = pd.to_numeric(api_df['tmdb_id'],    errors='coerce').astype('Int64')

# Pour Metacritic on joint sur le titre — pas sur tmdb_id
# car le scraping se fait par slug (nom du film), pas par ID
print('Types tmdb_id :')
print('  kaggle_df :', kaggle_df['tmdb_id'].dtype)
print('  api_df    :', api_df['tmdb_id'].dtype)

Types tmdb_id :
  kaggle_df : Int64
  api_df    : Int64


## 4. Identification des colonnes en commun

Avant de fusionner, on identifie les colonnes dupliquées pour éviter les conflits.

In [11]:
# Colonnes communes entre Kaggle et API (hors tmdb_id)
communes_api = set(kaggle_df.columns) & set(api_df.columns) - {'tmdb_id'}
print('Colonnes communes Kaggle / API :')
print(communes_api)

Colonnes communes Kaggle / API :
{'studio_country', 'title', 'release_date', 'runtime'}


In [12]:
# On garde uniquement les colonnes de l'API qui n'existent pas déjà dans Kaggle
# Plus les colonnes que l'API enrichit vraiment (studio, runtime, language)
cols_api_utiles = ['tmdb_id', 'studio', 'studio_country', 'runtime', 'language', 'status']

# Garder seulement celles qui existent dans api_df
cols_api_utiles = [c for c in cols_api_utiles if c in api_df.columns]
print('Colonnes API retenues pour la fusion :')
print(cols_api_utiles)

Colonnes API retenues pour la fusion :
['tmdb_id', 'studio', 'studio_country', 'runtime', 'language', 'status']


## 5. Fusion Kaggle + API (left join sur tmdb_id)

On utilise un **left join** pour garder tous les films Kaggle même si l'API n'a pas retourné de données pour certains.

In [13]:
# Fusion principale : Kaggle enrichi par l'API
df = kaggle_df.merge(
    api_df[cols_api_utiles],
    on='tmdb_id',
    how='left',
    suffixes=('', '_api')
)
print('Forme après fusion Kaggle + API :', df.shape)

Forme après fusion Kaggle + API : (3181, 35)


In [14]:
# Vérifier combien de films ont été enrichis par l'API
enrichis = df['studio'].notna().sum() if 'studio' in df.columns else 0
print(f'Films enrichis par l\'API : {enrichis} / {len(df)}')

Films enrichis par l'API : 255 / 3181


In [15]:
# Remplir les colonnes dupliquées : si Kaggle a déjà la valeur, on la garde
# Sinon on prend la valeur de l'API
if 'studio_name' in df.columns and 'studio' in df.columns:
    df['studio_name'] = df['studio_name'].fillna(df['studio'])
    df.drop(columns=['studio'], inplace=True)

if 'runtime_api' in df.columns:
    df['runtime'] = df['runtime'].fillna(df['runtime_api'])
    df.drop(columns=['runtime_api'], inplace=True)

print('Colonnes après nettoyage des doublons API :')
print(df.columns.tolist())

Colonnes après nettoyage des doublons API :
['tmdb_id', 'title', 'original_title', 'director', 'primary_genre', 'all_genres', 'original_language', 'primary_language', 'runtime', 'top_cast', 'overview', 'studio_name', 'studio_country', 'primary_country', 'release_date', 'release_year', 'release_month', 'release_quarter', 'release_day', 'release_week', 'release_season', 'budget', 'revenue', 'profit', 'roi', 'is_profit', 'revenue_tier', 'user_score', 'vote_count', 'popularity', 'studio_country_api', 'language', 'status']


## 6. Fusion avec les scores Metacritic (left join sur le titre)

Metacritic est jointé sur le titre normalisé car on n'a pas de tmdb_id côté Metacritic.

In [16]:
# Afficher les colonnes Metacritic disponibles
print(meta_df.head(10))

              slug  metascore  user_score
0  the-dark-knight        NaN         NaN
1        inception        NaN         NaN
2     interstellar        NaN         NaN
3           avatar        NaN         NaN


In [17]:
# Normaliser les titres pour la jointure : minuscules, sans espaces en trop
def normaliser_titre(titre):
    if pd.isna(titre):
        return ''
    return str(titre).lower().strip()

df['titre_normalise']       = df['title'].apply(normaliser_titre)
meta_df['titre_normalise']  = meta_df['slug'].apply(
    lambda s: str(s).replace('-', ' ').lower().strip()
)

print('Exemple titres normalisés Kaggle :')
print(df['titre_normalise'].head(5))
print('\nExemple titres normalisés Metacritic :')
print(meta_df['titre_normalise'].head(5))

Exemple titres normalisés Kaggle :
0      quantum of solace
1             robin hood
2     the golden compass
3           spider-man 2
4    monsters university
Name: titre_normalise, dtype: str

Exemple titres normalisés Metacritic :
0    the dark knight
1          inception
2       interstellar
3             avatar
Name: titre_normalise, dtype: str


In [18]:
# Fusion avec Metacritic sur le titre normalisé
df = df.merge(
    meta_df[['titre_normalise', 'metascore', 'user_score_meta']]
        if 'user_score_meta' in meta_df.columns
        else meta_df[['titre_normalise', 'metascore']],
    on='titre_normalise',
    how='left'
)
print('Forme après fusion Metacritic :', df.shape)

Forme après fusion Metacritic : (3181, 35)


In [19]:
# Vérifier combien de films ont un metascore
with_meta = df['metascore'].notna().sum()
print(f'Films avec Metascore : {with_meta} / {len(df)}')
print(df[['title', 'metascore']].dropna().head(10))

Films avec Metascore : 0 / 3181
Empty DataFrame
Columns: [title, metascore]
Index: []


In [20]:
# Supprimer la colonne de jointure temporaire
df.drop(columns=['titre_normalise'], inplace=True)
print('Colonne titre_normalise supprimée')

Colonne titre_normalise supprimée


## 7. Vérification finale du dataset fusionné

In [21]:
# Forme finale
print('Forme finale du dataset fusionné :', df.shape)

Forme finale du dataset fusionné : (3181, 34)


In [22]:
# Valeurs manquantes dans le dataset fusionné
valeurs_manquantes = df.isnull().sum()
print('Valeurs manquantes par colonne :')
print(valeurs_manquantes[valeurs_manquantes > 0])

Valeurs manquantes par colonne :
all_genres               1
primary_language         8
top_cast                 2
studio_country_api    2933
language              2926
status                2926
metascore             3181
dtype: int64


In [23]:
# Vérification des doublons dans le dataset fusionné
nb_doublons = df.duplicated(subset=['tmdb_id']).sum()
print(f'Doublons sur tmdb_id : {nb_doublons}')

Doublons sur tmdb_id : 0


In [24]:
# Supprimer les doublons éventuels introduits par la fusion
avant = len(df)
df.drop_duplicates(subset=['tmdb_id'], inplace=True)
print(f'Lignes supprimées (doublons post-fusion) : {avant - len(df)}')
print(f'Lignes restantes : {len(df)}')

Lignes supprimées (doublons post-fusion) : 0
Lignes restantes : 3181


In [25]:
# Aperçu final
print(df[['tmdb_id', 'title', 'director', 'primary_genre', 'studio_name',
          'revenue', 'profit', 'roi', 'user_score', 'metascore']].head(15))

    tmdb_id                                title         director  \
0     10764                    Quantum of Solace     Marc Forster   
1     20662                           Robin Hood     Ridley Scott   
2      2268                   The Golden Compass      Chris Weitz   
3       558                         Spider-Man 2        Sam Raimi   
4     62211                  Monsters University      Dan Scanlon   
5      8373  Transformers: Revenge of the Fallen      Michael Bay   
6     68728           Oz: The Great and Powerful        Sam Raimi   
7    102382             The Amazing Spider-Man 2        Marc Webb   
8     20526                         TRON: Legacy  Joseph Kosinski   
9     49013                               Cars 2    John Lasseter   
10    44912                        Green Lantern  Martin Campbell   
11      534                 Terminator Salvation              McG   
12    72190                          World War Z     Marc Forster   
13    54138              Star Trek

In [26]:
# Types des colonnes finales
print(df.dtypes)

tmdb_id                 Int64
title                     str
original_title            str
director                  str
primary_genre             str
all_genres                str
original_language         str
primary_language          str
runtime               float64
top_cast                  str
overview                  str
studio_name               str
studio_country            str
primary_country           str
release_date              str
release_year            int64
release_month           int64
release_quarter         int64
release_day             int64
release_week            int64
release_season            str
budget                  int64
revenue                 int64
profit                  int64
roi                   float64
is_profit                bool
revenue_tier              str
user_score            float64
vote_count              int64
popularity            float64
studio_country_api        str
language                  str
status                    str
metascore 

## 8. Sauvegarde du fichier fusionné

In [27]:
# Sauvegarde du dataset fusionné — prêt pour l'ETL pygramETL
df.to_csv('merged_movies.csv', index=False)
print(f'Fichier sauvegardé => merged_movies.csv ({len(df)} lignes)')

Fichier sauvegardé => merged_movies.csv (3181 lignes)


In [28]:
# Résumé final des sources
print('=== RÉSUMÉ DE LA FUSION ===')
print(f'Films issus de Kaggle         : {len(kaggle_df)}')
print(f'Films enrichis via API TMDB   : {api_df["tmdb_id"].isin(df["tmdb_id"]).sum()}')
print(f'Films avec score Metacritic   : {df["metascore"].notna().sum()}')
print(f'Total films dans merged_movies: {len(df)}')

=== RÉSUMÉ DE LA FUSION ===
Films issus de Kaggle         : 3181
Films enrichis via API TMDB   : 255
Films avec score Metacritic   : 0
Total films dans merged_movies: 3181
